# DistilBERT Fine-tuning — Toxic Comment Detection
**Antes de empezar**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4**

In [ ]:
# 1. Instalar dependencias
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# 2. Subir train.csv y val.csv desde tu PC
from google.colab import files
uploaded = files.upload()  # Sube train.csv y val.csv

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report

print('GPU disponible:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Cargar datos
train_df = pd.read_csv('train.csv')[['clean_text', 'IsToxic']].dropna()
val_df   = pd.read_csv('val.csv')[['clean_text', 'IsToxic']].dropna()

train_df = train_df.rename(columns={'clean_text': 'text', 'IsToxic': 'label'})
val_df   = val_df.rename(columns={'clean_text': 'text', 'IsToxic': 'label'})

print('Train:', len(train_df), '| Val:', len(val_df))
print('Distribución train:\n', train_df['label'].value_counts())

In [ ]:
# 4. Tokenizar
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df).map(tokenize, batched=True)

train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in ['input_ids','attention_mask','label']])
val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c not in ['input_ids','attention_mask','label']])

train_ds.set_format('torch')
val_ds.set_format('torch')

In [ ]:
# 5. Modelo
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
# 6. Métrica: F1 del tóxico (label=1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'f1_toxic': f1_score(labels, preds, pos_label=1)}

In [ ]:
# 7. Entrenamiento
training_args = TrainingArguments(
    output_dir='./distilbert_youtoxic',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_toxic',
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# 8. Evaluación final en val
preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=1)
print(classification_report(val_df['label'], preds, target_names=['No tóxico', 'Tóxico']))

In [ ]:
# 9. Guardar modelo y tokenizer
trainer.save_model('./distilbert_youtoxic')
tokenizer.save_pretrained('./distilbert_youtoxic')
print('Modelo guardado en ./distilbert_youtoxic')

In [ ]:
# 10. Descargar modelo como zip
import shutil
shutil.make_archive('distilbert_youtoxic', 'zip', '.', 'distilbert_youtoxic')
files.download('distilbert_youtoxic.zip')